# What does the player-share model predict?

Pick one Six Nations round and inspect its forecasts. The notebook fits on all
matches before that round, then compares player shares with and without
opposition against the recency-weighted mean. The squad comparison uses prices
when the prepared roster is available.

In [ ]:
from math import log
from pathlib import Path

import pandas as pd
from IPython.display import display

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.fantasy_fixtures import fixture_path, tournament_rounds
from six_nations_fantasy.data.load import DATA_DIR
from six_nations_fantasy.evaluation.pipeline import fit_predict, recorded_lineup
from six_nations_fantasy.evaluation.selection import prediction_lineup, scored_player_fixtures
from six_nations_fantasy.evaluation.squads import select_squad, squad_solver
from six_nations_fantasy.models import PlayerShareModel, RecencyWeightedMean
from six_nations_fantasy.models.player_share import component_units
from six_nations_fantasy.score import simple_score

data_dir = DATA_DIR   # Or Path("/path/to/your/data")
year, round_number = 2026, 1
tau = 0.25
stat = "tries"        # Choose a row from the component summary below.
team = None          # Show every side, or set a name such as "France".

In [ ]:
dataset = load_dataset(data_dir)
schedule = dict(tournament_rounds(dataset.matches, year))[round_number]
round_start = schedule["date"].min()
training = dataset.matches_before(round_start)
held = dataset.select_matches(schedule["match_id"])

roster_file = fixture_path(data_dir, year)
roster = pd.read_parquet(roster_file) if roster_file.exists() else None
if roster is not None and not roster["round"].eq(round_number).any():
    roster = None
scored = scored_player_fixtures(held, round_number, roster)
fixtures = prediction_lineup(scored)
print(f"Training cutoff {round_start.date()} | {len(training.matches):,} matches")
print(f"{len(fixtures)} players from {'announced rosters' if roster is not None else 'recorded team sheets'}")

scored = scored.copy()
if "name" not in scored:
    registry_file = data_dir / "players.parquet"
    if registry_file.exists():
        registry = pd.read_parquet(registry_file)
        if "player_id" in registry:
            registry = registry.set_index("player_id")
        scored["name"] = scored["player_id"].map(registry["name"])
    else:
        scored["name"] = scored["player_id"].astype(str)

In [ ]:
share = PlayerShareModel(tau=tau)
without_opposition = PlayerShareModel(tau=0)
recency = RecencyWeightedMean(scorer=simple_score, half_life_days=7 * log(0.5) / log(0.85))

comparison = {
    "share": fit_predict(share, training, fixtures),
    "share_no_opposition": fit_predict(without_opposition, training, fixtures),
    "recency": fit_predict(recency, training, fixtures),
}
forecasts = pd.DataFrame(comparison).assign(actual=scored["actual"])

## Team multipliers across the scoring stats

For each stat, the team forecast is the historical starting average times the
competition, home/away, team and opponent multipliers. The table shows how big
the fitted opponent multipliers are in this round. It also compares the two
**refitted** team forecasts. Their difference can include changes in the team
and competition coefficients when the opponent term is removed.

In [ ]:
keys = ["match_id", "team"]
recorded = recorded_lineup(held)
actual_counts = component_units(held.performances)
actual_team = actual_counts.groupby([recorded[key].to_numpy() for key in keys]).sum()
actual_team.index.names = keys

team_tables = {}
for component in share.opponent_scales:
    with_opp, _ = share.explain_component(fixtures, component)
    without_opp, _ = without_opposition.explain_component(fixtures, component)
    team_tables[component] = (
        with_opp.merge(
            without_opp[keys + ["expected_units"]].rename(
                columns={"expected_units": "expected_without_opposition"}
            ),
            on=keys, validate="one_to_one",
        )
        .join(actual_team[component].rename("observed_units"), on=keys)
        .assign(change_with_opposition=lambda rows:
                rows["expected_units"] - rows["expected_without_opposition"])
    )

summary = {}
for component, sides in team_tables.items():
    with_error = sides["expected_units"] - sides["observed_units"]
    without_error = sides["expected_without_opposition"] - sides["observed_units"]
    summary[component] = {
        "smallest opponent multiplier": sides["opponent_multiplier"].min(),
        "largest opponent multiplier": sides["opponent_multiplier"].max(),
        "mean absolute change in team units": sides["change_with_opposition"].abs().mean(),
        "team MAE with opposition": with_error.abs().mean(),
        "team MAE without opposition": without_error.abs().mean(),
    }
display(pd.DataFrame.from_dict(summary, orient="index").round(3))
display(team_tables[stat].drop(columns=["date", "competition"]).round(3))

## How the team total is split

Multiply the position weight by the player multiplier, then divide by the total
weight on that sheet. The resulting share is the fraction of the team's units
assigned to the player. Observed shares use the recorded team total. A team with
zero observed units has no observed shares. Metres in these tables are raw
metres, with negative values counted as zero.

In [ ]:
_, allocation = share.explain_component(fixtures, stat)
actual_players = recorded[["match_id", "player_id"]].assign(
    observed_units=actual_counts[stat].to_numpy()
)
allocation = (
    allocation.assign(name=scored["name"].to_numpy())
    .merge(actual_players, on=["match_id", "player_id"], how="left", validate="one_to_one")
    .join(actual_team[stat].rename("observed_team_units"), on=keys)
)
allocation["observed_share"] = (
    allocation["observed_units"].fillna(0)
    / allocation["observed_team_units"].where(allocation["observed_team_units"].ne(0))
)
columns = [
    "team", "name", "number", "position_role", "position_weight", "player_multiplier",
    "share", "observed_share", "expected_units", "observed_units",
]
selected = allocation if team is None else allocation.loc[allocation["team"].eq(team)]
display(selected.sort_values(["team", "share"], ascending=[True, False])[columns].round(3))

## Does the model pick different players?

The first table compares player errors and rank correlation on this round. The
next shows where the share model and recency mean disagree most. When prices are
available, solve the same fantasy squad problem for each forecast and compare
the players selected, captain, supersub and points earned.

In [ ]:
model_names = list(comparison)
errors = forecasts[model_names].sub(forecasts["actual"], axis=0)
ranks = forecasts[model_names].rank()
display(pd.DataFrame({
    "player RMSE": errors.pow(2).mean().pow(0.5),
    "rank correlation with recency": ranks.corr()["recency"],
}).round(3))

details = scored[["match_id", "team", "player_id", "name"]].join(forecasts)
details["effect of refitting with opposition"] = details["share"] - details["share_no_opposition"]
details["share minus recency"] = details["share"] - details["recency"]
order = details["share minus recency"].abs().sort_values(ascending=False).index
display(details.loc[order].head(15).round(2))

In [ ]:
if roster is not None:
    solver = squad_solver()
    candidates = scored.join(forecasts[model_names]).dropna(subset=["cost"])
    squads = {name: select_squad(candidates, name, solver) for name in [*model_names, "actual"]}
    recency_ids = set(squads["recency"]["player_id"])
    optimum = squads["actual"]["actual_total"].sum()
    rows = []
    for name, squad in squads.items():
        actual = squad["actual_total"].sum()
        rows.append({
            "selection": "hindsight optimum" if name == "actual" else name,
            "players also picked by recency": len(recency_ids & set(squad["player_id"])),
            "captain": squad.loc[squad["captain"], "name"].iloc[0],
            "supersub": squad.loc[squad["supersub"], "name"].iloc[0],
            "actual points": actual,
            "% of optimum": 100 * actual / optimum if optimum else float("nan"),
        })
    display(pd.DataFrame(rows).round(2))
else:
    print("Prepare this round's fantasy roster to compare squad selections.")